# Python Descriptors: 12 New Advanced Problems with Worked Solutions

### A tutorial-style continuation of the supplied `Int` / `Float` / `List` → `ValidType` lesson

**Audience:** intermediate-to-advanced Python learners. **Runtime:** Python 3.10+; Python standard library only. **Format:** problem → small experiment → explanation → incremental solution → edge-case tests → design review. Every problem introduces its own implementation instead of relying on the previous tutorial notebook.

The supplied lesson begins with three repeated validators, extracts a reusable `ValidType`, binds attribute names with `__set_name__`, and broadens a float check to `numbers.Real`. Here we explore *different* follow-on questions: repeated descriptor objects, normalization pipelines, introspection, extensible validators, explicit slotted storage, state machines, monetary precision, safe mutable sequences, concurrency, migration, delegated storage, and multiple inheritance. These examples are newly constructed extensions, not claims that these scenarios appeared in the original material.

**How to use this notebook:** run cells from top to bottom; pause at each question; review the explanation after each experiment. Assertions are part of the solution. No pip installation or network connection is needed.

## A tiny testing vocabulary

We keep expected failures visible without allowing a deliberately invalid operation to stop the notebook. `raises` checks an exception's type and optionally part of its message. Unexpected exceptions and missing exceptions both fail the cell. This is a local teaching helper, not a replacement for a full test framework.

In [1]:
def raises(error_type, operation, message=None):
    try:
        operation()
    except error_type as exc:
        if message is not None:
            assert message in str(exc), (message, str(exc))
        print(f"Expected {error_type.__name__}: {exc}")
        return exc
    except Exception as exc:
        raise AssertionError(f"Expected {error_type.__name__}, got {type(exc).__name__}") from exc
    raise AssertionError(f"Expected {error_type.__name__}, but no exception was raised")

print('Test helper is ready.')

Test helper is ready.


### The essential rule we will keep revisiting

A descriptor is a class attribute implementing some of `__get__`, `__set__`, and `__delete__`. Python binds `__set_name__(owner, name)` when it creates a class containing that descriptor in its class body. A descriptor that implements `__set__` or `__delete__` is a **data descriptor**. We normally store actual field values on each *instance*; class-level descriptor objects store reusable rules and binding metadata.

A clean setter typically computes and checks a proposed value **before** touching persistent state. We will deliberately test what happens when that rule is violated or incomplete.

## Problem 1 — One descriptor object, two attribute names

**The challenge.** A programmer wants to reuse a single descriptor instance under two names, then reuse it in a second class. Why can this silently mix up field names, and how can class creation repair it?

**Success criteria.** Give each declared attribute a distinct descriptor object; retain independent per-instance values; support reuse across different owner classes; keep validators reusable.

We will solve it through small experiments. Try to predict each outcome *before* executing its cell, then compare it with the explanation that follows.

### Step 1 — Make the source of the bug obvious

The descriptor needs precisely one `(owner, name)` pair for storing values. Here we *observe* the two naming callbacks. A single object cannot remember both names in one `self.name`.

In [2]:
class NaiveNamed:
    def __set_name__(self, owner, name):
        print(f"binding {name} on {owner.__name__}")
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__.get(self.name)

    def __set__(self, instance, value):
        instance.__dict__[self.name] = value

shared_naive = NaiveNamed()
class BrokenPair:
    left = shared_naive
    right = shared_naive

broken = BrokenPair()
broken.left = 5
assert broken.right == 5
print('Stored dictionary:', broken.__dict__)

binding left on BrokenPair
binding right on BrokenPair
Stored dictionary: {'right': 5}


**Observation.** The name was bound twice, and the *last* binding won. The assignment to `left` used the key `right`, so `left` and `right` unintentionally became aliases. Creating separate instances manually works, but a reusable abstraction can protect class authors from this subtle mistake.

### Step 2 — Define a descriptor that records its owner and validates assignments

This descriptor deliberately refuses rebinding. We will make a metaclass clone an already-bound or duplicated descriptor **before** Python invokes `__set_name__` for the new class. The validator is the same callable in each clone; field binding is different.

In [3]:
class NamedField:
    def __init__(self, validator):
        self.validator = validator
        self.owner = None
        self.name = None

    def __set_name__(self, owner, name):
        if self.owner is not None:
            raise RuntimeError('A NamedField cannot be bound twice')
        self.owner, self.name = owner, name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        try:
            return instance.__dict__[self.name]
        except KeyError:
            raise AttributeError(f'{self.name} has not been set') from None

    def __set__(self, instance, value):
        self.validator(value)
        instance.__dict__[self.name] = value


def strict_integer(value):
    if type(value) is not int:  # bool is intentionally excluded
        raise TypeError('expected an exact int')

### Step 3 — Copy duplicates in a metaclass namespace

The metaclass inspects the proposed class dictionary before the class is created. `copy.copy` is sufficient here because the descriptor contains an immutable validator reference and simple binding fields; if a descriptor held mutable configuration, cloning would need a more deliberate policy.

In [4]:
import copy

class CloneFieldsMeta(type):
    def __new__(metacls, name, bases, namespace):
        namespace = dict(namespace)
        seen = set()
        for key, value in namespace.items():
            if not isinstance(value, NamedField):
                continue
            if id(value) in seen or value.owner is not None:
                value = copy.copy(value)
                value.owner = None
                value.name = None
                namespace[key] = value
            seen.add(id(value))
        return super().__new__(metacls, name, bases, namespace)

shared = NamedField(strict_integer)
class Pair(metaclass=CloneFieldsMeta):
    left = shared
    right = shared

class AnotherPair(metaclass=CloneFieldsMeta):
    item = shared

assert Pair.left is not Pair.right
assert Pair.left is not AnotherPair.item
print([(fld.owner.__name__, fld.name) for fld in (Pair.left, Pair.right, AnotherPair.item)])

[('Pair', 'left'), ('Pair', 'right'), ('AnotherPair', 'item')]


### Step 4 — Test isolation in both dimensions

We need **field isolation** within a class and **instance isolation** across objects. We also verify failure atomicity: an invalid value must not erase a valid existing value.

In [5]:
a, b = Pair(), Pair()
a.left, a.right, b.left = 10, 20, 30
assert (a.left, a.right, b.left) == (10, 20, 30)
assert a.__dict__ == {'left': 10, 'right': 20}
raises(TypeError, lambda: setattr(a, 'left', True), 'exact int')
assert a.left == 10
other = AnotherPair()
other.item = 99
assert other.item == 99
print('Both fields and both instances are independent.')

Expected TypeError: expected an exact int
Both fields and both instances are independent.


**Design review.** Metaclasses are justified only if this safety is worth the added complexity. The simpler API `left = NamedField(...); right = NamedField(...)` avoids cloning entirely. Our metaclass cannot solve every exotic descriptor with custom binding or deep mutable internals; treat cloning as an explicit contract for `NamedField`, not a universal trick.

## Problem 2 — A transactional text-normalization pipeline

**The challenge.** A `ValidType(str)` field checks its type but does not make names consistent. Create a descriptor that transforms Unicode input, validates the transformed result, and stores nothing if any stage fails.

**Success criteria.** Support reusable normalizers/validators, reject non-string inputs before normalization, preserve existing state on failure, and keep field-specific error messages.

We will solve it through small experiments. Try to predict each outcome *before* executing its cell, then compare it with the explanation that follows.

### Step 1 — Understand why the order matters

A decomposed Unicode string may look identical to its composed form; whitespace and case may also hide duplicates. Normalize **before** enforcing length and non-empty rules, so validation describes the value that will actually be stored.

In [6]:
import unicodedata

raw = '  Cafe\u0301  '
composed = unicodedata.normalize('NFC', raw).strip().casefold()
assert composed == 'café'
print('Before:', repr(raw), 'After:', repr(composed))

Before: '  Café  ' After: 'café'


### Step 2 — Implement distinct, composable stages

Each normalizer returns a new candidate; each validator either returns normally or raises. `__set__` uses a local variable throughout the pipeline, and changes the instance dictionary only at the very end. This avoids partially applied updates.

In [7]:
class TextPipeline:
    def __init__(self, *, normalizers=(), validators=()):
        self.normalizers = tuple(normalizers)
        self.validators = tuple(validators)

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        if self.name not in instance.__dict__:
            raise AttributeError(f'{self.name} is unset')
        return instance.__dict__[self.name]

    def __set__(self, instance, value):
        if not isinstance(value, str):
            raise TypeError(f'{self.name} must be a string')
        candidate = value
        for normalize in self.normalizers:
            candidate = normalize(candidate)
        for check in self.validators:
            check(candidate)
        instance.__dict__[self.name] = candidate


def nonblank(text):
    if not text:
        raise ValueError('value cannot be blank')


def at_most(limit):
    def check(text):
        if len(text) > limit:
            raise ValueError(f'length cannot exceed {limit}')
    return check


def nfc(text):
    return unicodedata.normalize('NFC', text)

### Step 3 — Configure two fields without duplicating setter code

A display name should retain case, while a searchable handle should use `casefold()`. The descriptor class remains the same; its configuration expresses different requirements.

In [8]:
class ProfileText:
    display_name = TextPipeline(
        normalizers=(nfc, str.strip), validators=(nonblank, at_most(25)))
    handle = TextPipeline(
        normalizers=(nfc, str.strip, str.casefold), validators=(nonblank, at_most(12)))

profile = ProfileText()
profile.display_name = '  José  '
profile.handle = '  CAFE\u0301  '
assert (profile.display_name, profile.handle) == ('José', 'café')
print(profile.__dict__)

{'display_name': 'José', 'handle': 'café'}


### Step 4 — Check failures at different pipeline stages

The original value survives each rejection. Think about what would go wrong if we had written into `__dict__` after `strip()` but before the length check.

In [9]:
raises(TypeError, lambda: setattr(profile, 'handle', 123), 'string')
raises(ValueError, lambda: setattr(profile, 'handle', '   '), 'blank')
raises(ValueError, lambda: setattr(profile, 'handle', 'X' * 20), '12')
assert profile.handle == 'café'
print('Previous state survived all three failures.')

Expected TypeError: handle must be a string
Expected ValueError: value cannot be blank
Expected ValueError: length cannot exceed 12
Previous state survived all three failures.


**Design review.** A normalizer should ideally be deterministic and side-effect free. A validator should not mutate an instance behind the pipeline's back. This example handles validation atomicity, not transactions across *multiple* fields or external resources.

## Problem 3 — Missing values, `hasattr`, and safe introspection

**The challenge.** A descriptor returns `None` for an unset field. How can a caller tell “missing” apart from “explicitly assigned `None`”? How can tooling inspect the descriptor object without triggering its getter?

**Success criteria.** Use standard missing-attribute semantics, show `hasattr` and `getattr` behavior, and contrast dynamic lookup with `inspect.getattr_static`.

We will solve it through small experiments. Try to predict each outcome *before* executing its cell, then compare it with the explanation that follows.

### Step 1 — Demonstrate an ambiguous getter

Using `instance.__dict__.get(name, None)` erases the distinction between a missing key and a present key whose value is `None`. Both look identical to normal reads.

In [10]:
class Ambiguous:
    def __get__(self, obj, owner):
        return None if obj is not None else self

class Placeholder:
    value = Ambiguous()

placeholder = Placeholder()
assert placeholder.value is None
assert hasattr(placeholder, 'value')  # even though nothing was stored
print('hasattr reported:', hasattr(placeholder, 'value'))

hasattr reported: True


### Step 2 — Follow the ordinary Python attribute contract

`AttributeError` signals an absent attribute. This lets `hasattr` return `False` and `getattr(obj, name, default)` supply its fallback. We intentionally allow explicit `None`, which is now distinguishable from unset.

In [11]:
class OptionalField:
    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        try:
            return instance.__dict__[self.name]
        except KeyError:
            raise AttributeError(self.name) from None

    def __set__(self, instance, value):
        instance.__dict__[self.name] = value

class Envelope:
    payload = OptionalField()

env = Envelope()
assert not hasattr(env, 'payload')
assert getattr(env, 'payload', '<missing>') == '<missing>'
raises(AttributeError, lambda: env.payload, 'payload')
env.payload = None
assert hasattr(env, 'payload') and env.payload is None
print('Missing and explicit None are different states.')

Expected AttributeError: payload
Missing and explicit None are different states.


### Step 3 — See why ordinary introspection can execute code

Ordinary `getattr` invokes `__get__`. Tooling may instead need the raw class attribute, without executing validation, caching, or side effects. `inspect.getattr_static` helps do that, but it returns the raw descriptor rather than its computed value.

In [12]:
import inspect

class ReadCounter:
    def __set_name__(self, owner, name):
        self.name = name
        self.reads = 0

    def __get__(self, obj, owner):
        if obj is None:
            return self
        self.reads += 1
        return obj.__dict__[self.name]

    def __set__(self, obj, value):
        obj.__dict__[self.name] = value

class Sensor:
    temperature = ReadCounter()

sensor = Sensor()
sensor.temperature = 18
raw_descriptor = inspect.getattr_static(sensor, 'temperature')
assert raw_descriptor is Sensor.temperature
assert Sensor.temperature.reads == 0
assert sensor.temperature == 18
assert Sensor.temperature.reads == 1
print('Static inspection did not increment reads.')

Static inspection did not increment reads.


**Design review.** `inspect.getattr_static` avoids dynamic lookup, but its results may include slot descriptors and other low-level objects. It is an introspection tool, not a transparent replacement for `getattr`. Do not catch every exception in `__get__` and turn it into `AttributeError`: doing so can hide actual programming errors.

## Problem 4 — Extend validation without repeating `__set__`

**The challenge.** A base integer descriptor checks types. Specialized fields must enforce positivity, parity, or a configurable divisor. How can subclasses extend validation while preserving one canonical storage algorithm?

**Success criteria.** Provide a template-method validator, use `super()` in stricter rules, reject `bool`, and prove failed assignments do not modify state.

We will solve it through small experiments. Try to predict each outcome *before* executing its cell, then compare it with the explanation that follows.

### Step 1 — Put the shared algorithm in a base class

The descriptor coordinates validation and storage; subclasses override only the `validate` hook. This is a useful alternative to copying an entire setter for each new domain constraint.

In [13]:
class IntegerField:
    def __set_name__(self, owner, name):
        self.name = name

    def validate(self, value):
        if type(value) is not int:
            raise TypeError(f'{self.name} must be an int, not bool or another type')
        return value

    def __get__(self, instance, owner):
        if instance is None:
            return self
        if self.name not in instance.__dict__:
            raise AttributeError(self.name)
        return instance.__dict__[self.name]

    def __set__(self, instance, value):
        candidate = self.validate(value)
        instance.__dict__[self.name] = candidate

class PositiveInteger(IntegerField):
    def validate(self, value):
        value = super().validate(value)
        if value <= 0:
            raise ValueError(f'{self.name} must be positive')
        return value

### Step 2 — Add another layer instead of forking the setter

`EvenPositive` reuses both its ancestors. `MultipleOf` adds per-field configuration without creating a new descriptor class for every divisor.

In [14]:
class EvenPositive(PositiveInteger):
    def validate(self, value):
        value = super().validate(value)
        if value % 2:
            raise ValueError(f'{self.name} must be even')
        return value

class MultipleOf(PositiveInteger):
    def __init__(self, divisor):
        if type(divisor) is not int or divisor <= 0:
            raise ValueError('divisor must be a positive int')
        self.divisor = divisor

    def validate(self, value):
        value = super().validate(value)
        if value % self.divisor:
            raise ValueError(f'{self.name} must be divisible by {self.divisor}')
        return value

class ManufacturingOrder:
    quantity = PositiveInteger()
    carton_size = EvenPositive()
    batch = MultipleOf(5)

### Step 3 — Exercise each branch of the validation hierarchy

We test both a successful write and one or more rejected writes per field. Assertions prove the shared setter only commits after the deepest validation hook completes.

In [15]:
order = ManufacturingOrder()
order.quantity, order.carton_size, order.batch = 3, 8, 15
raises(TypeError, lambda: setattr(order, 'quantity', True), 'int')
raises(ValueError, lambda: setattr(order, 'quantity', 0), 'positive')
raises(ValueError, lambda: setattr(order, 'carton_size', 7), 'even')
raises(ValueError, lambda: setattr(order, 'batch', 12), 'divisible')
assert (order.quantity, order.carton_size, order.batch) == (3, 8, 15)
print(order.__dict__)

Expected TypeError: quantity must be an int, not bool or another type
Expected ValueError: quantity must be positive
Expected ValueError: carton_size must be even
Expected ValueError: batch must be divisible by 5
{'quantity': 3, 'carton_size': 8, 'batch': 15}


**Design review.** It is possible for a subclass to override `__set__` and bypass the hook, so this is a coding convention rather than an enforcement mechanism. For highly controlled APIs, consider composition of callable validators, as in Problem 2. The two patterns solve related but different extensibility needs.

## Problem 5 — Store descriptor values in explicit `__slots__`

**The challenge.** The original `ValidType` writes to `instance.__dict__`, which fails on ordinary slotted instances. Can we support a slotted owner without an external weak-key mapping?

**Success criteria.** Reserve a private slot for each field, validate before writing it, support missing reads/deletion, and keep the instance free of `__dict__`.

We will solve it through small experiments. Try to predict each outcome *before* executing its cell, then compare it with the explanation that follows.

### Step 1 — Observe the incompatibility

A class declaring `__slots__ = ()` generally has no instance dictionary. Direct dictionary storage cannot work. Here the error is expected; the next solution uses a different, explicit storage location.

In [16]:
class NoDictionary:
    __slots__ = ()

slotted = NoDictionary()
assert not hasattr(slotted, '__dict__')
raises(AttributeError, lambda: slotted.__dict__)
print('A slotted instance does not provide dictionary storage.')

Expected AttributeError: 'NoDictionary' object has no attribute '__dict__'
A slotted instance does not provide dictionary storage.


### Step 2 — Keep the public descriptor separate from its private slot

We reserve `_temperature` as a slot. That class attribute is automatically a slot descriptor, while `temperature` is *our* validating descriptor. Accessing `_temperature` does not re-enter `temperature`, so delegation is safe.

In [17]:
import math
import numbers

class SlottedReal:
    def __init__(self, storage_slot):
        self.storage_slot = storage_slot

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        try:
            return object.__getattribute__(instance, self.storage_slot)
        except AttributeError:
            raise AttributeError(f'{self.name} is unset') from None

    def __set__(self, instance, value):
        if isinstance(value, bool) or not isinstance(value, numbers.Real):
            raise TypeError(f'{self.name} requires a real number (not bool)')
        candidate = float(value)
        if not math.isfinite(candidate):
            raise ValueError(f'{self.name} must be finite')
        object.__setattr__(instance, self.storage_slot, candidate)

    def __delete__(self, instance):
        object.__delattr__(instance, self.storage_slot)

class Thermometer:
    __slots__ = ('_temperature',)
    temperature = SlottedReal('_temperature')

### Step 3 — Test the entire slotted lifecycle

The descriptor reads/writes a private slot, can delete its value, and still uses normal `AttributeError` semantics for an unset field. Validation precedes conversion and storage; infinities and booleans are excluded.

In [18]:
thermo = Thermometer()
assert not hasattr(thermo, '__dict__')
assert not hasattr(thermo, 'temperature')
thermo.temperature = 21
assert thermo.temperature == 21.0
raises(TypeError, lambda: setattr(thermo, 'temperature', True), 'not bool')
raises(ValueError, lambda: setattr(thermo, 'temperature', float('inf')), 'finite')
assert thermo.temperature == 21.0
del thermo.temperature
assert not hasattr(thermo, 'temperature')
print('Validated, stored, and deleted without __dict__.')

Expected TypeError: temperature requires a real number (not bool)
Expected ValueError: temperature must be finite
Validated, stored, and deleted without __dict__.


**Design review.** Every public field needs a distinct reserved storage slot. `object.__setattr__` does not bypass the slot's own normal checks or create arbitrary slots. This solution requires control over the owner class declaration; it does **not** transparently retrofit arbitrary pre-existing slotted types. `numbers.Real` does not cover every numeric type (notably `Decimal`), so Problem 7 uses an explicit precision policy.

## Problem 6 — A descriptor as a finite-state machine

**The challenge.** Type checking does not prevent a workflow from jumping from `draft` directly to `approved`. Make the descriptor enforce *allowed state transitions*, not merely allowed strings.

**Success criteria.** Specify initial states and legal transitions, reject unknown states and illegal jumps, preserve state on failure, and make deletion impossible.

We will solve it through small experiments. Try to predict each outcome *before* executing its cell, then compare it with the explanation that follows.

### Step 1 — Describe the policy as data before writing control flow

A transition table is easier to review than a long nested `if` statement. `None` represents an uninitialized instance here; it is not a real workflow state. This lets the first assignment have a separate rule.

In [19]:
TRANSITIONS = {
    None: {'draft'},
    'draft': {'submitted'},
    'submitted': {'approved', 'rejected'},
    'rejected': {'draft'},
    'approved': set(),
}
assert 'approved' not in TRANSITIONS['draft']
print('Legal next states from submitted:', sorted(TRANSITIONS['submitted']))

Legal next states from submitted: ['approved', 'rejected']


### Step 2 — Put the transition check inside the descriptor

The descriptor reads the previous state for this instance, verifies the proposed state, and commits exactly once. It never stores workflow state on itself: different orders share rules, not progress.

In [20]:
class StateField:
    def __init__(self, transition_table):
        self.transitions = {key: frozenset(values)
                            for key, values in transition_table.items()}

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        if self.name not in instance.__dict__:
            raise AttributeError(self.name)
        return instance.__dict__[self.name]

    def __set__(self, instance, proposed):
        if proposed not in self.transitions:
            raise ValueError(f'unknown state: {proposed!r}')
        old = instance.__dict__.get(self.name, None)
        if proposed not in self.transitions[old]:
            raise ValueError(f'illegal transition: {old!r} -> {proposed!r}')
        instance.__dict__[self.name] = proposed

    def __delete__(self, instance):
        raise AttributeError(f'{self.name} cannot be deleted')

class ReviewRequest:
    status = StateField(TRANSITIONS)

    def __init__(self):
        self.status = 'draft'

### Step 3 — Run a legal path and attempt illegal shortcuts

Both instances use the *same class-level descriptor*, yet they move independently. Failed transitions preserve the old state. We also check that a completed request cannot restart without an explicit policy change.

In [21]:
first, second = ReviewRequest(), ReviewRequest()
raises(ValueError, lambda: setattr(first, 'status', 'approved'), 'illegal transition')
first.status = 'submitted'
first.status = 'rejected'
first.status = 'draft'
first.status = 'submitted'
first.status = 'approved'
assert first.status == 'approved' and second.status == 'draft'
raises(ValueError, lambda: setattr(first, 'status', 'draft'), 'illegal transition')
raises(ValueError, lambda: setattr(second, 'status', 'missing'), 'unknown state')
raises(AttributeError, lambda: delattr(first, 'status'), 'cannot be deleted')
print('Independent requests:', first.status, second.status)

Expected ValueError: illegal transition: 'draft' -> 'approved'
Expected ValueError: illegal transition: 'approved' -> 'draft'
Expected ValueError: unknown state: 'missing'
Expected AttributeError: status cannot be deleted
Independent requests: approved draft


**Design review.** State machine descriptors work well for in-process models, but direct dictionary mutation can bypass the public attribute contract. They are not an authorization or audit system. If multiple threads update one workflow concurrently, protect the whole read-check-write transition with the same instance lock, using the principle in Problem 9.

## Problem 7 — Money without binary-float surprises

**The challenge.** A validator based on `numbers.Real` accepts ordinary floats, but floats are often inappropriate for exact monetary amounts. Design a descriptor storing nonnegative two-decimal-place `Decimal` values.

**Success criteria.** Accept decimal strings, exact integers, and `Decimal`; reject `bool`, floats, NaN, infinity, negative values, and excess nonzero fractional precision; do not silently round.

We will solve it through small experiments. Try to predict each outcome *before* executing its cell, then compare it with the explanation that follows.

### Step 1 — See the representation and input-type issue

A binary float may already encode an approximation before our descriptor sees it. Instead of converting `float` through `Decimal(float)`, we reject floats and ask callers to supply explicit decimal input.

In [22]:
from decimal import Decimal, InvalidOperation

print('Decimal from literal float:', Decimal(0.1))
print('Decimal from string:', Decimal('0.1'))
assert Decimal(0.1) != Decimal('0.1')

Decimal from literal float: 0.1000000000000000055511151231257827021181583404541015625
Decimal from string: 0.1


### Step 2 — Centralize conversion and exact precision checks

Quantization can silently round. For this exercise, a value is valid only if quantizing to cents leaves its numeric value unchanged. We also check `is_finite` *before* comparisons to avoid problematic exceptional values.

In [23]:
CENT = Decimal('0.01')

class MoneyField:
    def __set_name__(self, owner, name):
        self.name = name

    def _parse(self, value):
        if isinstance(value, bool) or isinstance(value, float):
            raise TypeError(f'{self.name}: use Decimal, int, or decimal string')
        if not isinstance(value, (Decimal, int, str)):
            raise TypeError(f'{self.name}: unsupported monetary input')
        try:
            amount = Decimal(value)
        except (InvalidOperation, ValueError):
            raise ValueError(f'{self.name}: invalid decimal amount') from None
        if not amount.is_finite():
            raise ValueError(f'{self.name}: amount must be finite')
        if amount < 0:
            raise ValueError(f'{self.name}: amount cannot be negative')
        try:
            cents = amount.quantize(CENT)
        except InvalidOperation:
            raise ValueError(f'{self.name}: amount is outside decimal precision') from None
        if cents != amount:
            raise ValueError(f'{self.name}: at most two nonzero decimal places')
        return cents

    def __get__(self, instance, owner):
        if instance is None:
            return self
        if self.name not in instance.__dict__:
            raise AttributeError(self.name)
        return instance.__dict__[self.name]

    def __set__(self, instance, value):
        instance.__dict__[self.name] = self._parse(value)

class InvoiceLine:
    unit_price = MoneyField()

### Step 3 — Verify exact arithmetic and acceptable equivalent scales

`"1.230"` has three decimal digits in its representation but equals exactly `1.23`; it is accepted because no nonzero precision is lost. We store the quantized `Decimal` for a consistent two-place representation.

In [24]:
line = InvoiceLine()
line.unit_price = '1.230'
assert line.unit_price == Decimal('1.23')
assert str(line.unit_price) == '1.23'
line.unit_price = 2
assert line.unit_price == Decimal('2.00')
total = line.unit_price * 3
assert total == Decimal('6.00')
print('Exact total:', total)

Exact total: 6.00


### Step 4 — Attack the boundaries, then check rollback

A valid current price must survive invalid inputs. We check common money hazards individually so that a failed assertion identifies the policy that broke.

In [25]:
for bad, error, fragment in [
    (True, TypeError, 'Decimal'),
    (0.1, TypeError, 'Decimal'),
    ('NaN', ValueError, 'finite'),
    ('Infinity', ValueError, 'finite'),
    ('-0.01', ValueError, 'negative'),
    ('1.235', ValueError, 'two nonzero'),
    ('oops', ValueError, 'invalid decimal'),
]:
    raises(error, lambda value=bad: setattr(line, 'unit_price', value), fragment)
assert line.unit_price == Decimal('2.00')
print('No invalid price was committed.')

Expected TypeError: unit_price: use Decimal, int, or decimal string
Expected TypeError: unit_price: use Decimal, int, or decimal string
Expected ValueError: unit_price: amount must be finite
Expected ValueError: unit_price: amount must be finite
Expected ValueError: unit_price: amount cannot be negative
Expected ValueError: unit_price: at most two nonzero decimal places
Expected ValueError: unit_price: invalid decimal amount
No invalid price was committed.


**Design review.** Real money systems also need currency codes, tax rules, configured precision, and a defined rounding policy *where rounding is actually intended*. This field intentionally refuses silent rounding. `Decimal` has a configurable arithmetic context; this tutorial assumes ordinary reasonable-sized prices, not arbitrary huge exponents.

## Problem 8 — Close the mutable-value loophole

**The challenge.** A `ValidType(list)` descriptor validates `obj.tags = [...]`, but `obj.tags.append(42)` never calls its setter. Build a typed list view that validates mutations, including multi-element changes atomically.

**Success criteria.** Validate append/insert/index assignment/slice assignment/extend; keep instances isolated; reject an invalid bulk update before changing any elements; avoid aliasing the assigned input list.

We will solve it through small experiments. Try to predict each outcome *before* executing its cell, then compare it with the explanation that follows.

### Step 1 — Prove that plain list validation is incomplete

A descriptor only sees normal assignment to its *managed attribute*. Once it returns a raw mutable list, mutations are methods on that list, not attribute assignments. The next implementation returns a controlled sequence object instead.

In [26]:
class ShallowListField:
    def __set_name__(self, owner, name):
        self.name = name
    def __get__(self, obj, owner):
        return self if obj is None else obj.__dict__[self.name]
    def __set__(self, obj, value):
        if not isinstance(value, list) or not all(isinstance(x, str) for x in value):
            raise TypeError('expected list[str]')
        obj.__dict__[self.name] = value

class ShallowDocument:
    tags = ShallowListField()

shallow = ShallowDocument()
shallow.tags = ['python']
shallow.tags.append(99)
assert shallow.tags == ['python', 99]
print('The ordinary setter did not see append:', shallow.tags)

The ordinary setter did not see append: ['python', 99]


### Step 2 — Build a controlled `MutableSequence`

`collections.abc.MutableSequence` supplies convenience methods in terms of a small core. We override `extend` separately because its default repeated insertion could leave a partial update when a later item is invalid. For slice assignment, validate the entire replacement before touching the list.

In [27]:
from collections.abc import MutableSequence

class CheckedStringList(MutableSequence):
    def __init__(self, values=()):
        self._items = []
        self.extend(values)

    @staticmethod
    def _validate(value):
        if not isinstance(value, str):
            raise TypeError('all elements must be strings')
        return value

    def __len__(self):
        return len(self._items)

    def __getitem__(self, index):
        return self._items[index]

    def __setitem__(self, index, value):
        if isinstance(index, slice):
            proposed = [self._validate(item) for item in value]
            self._items[index] = proposed
        else:
            self._items[index] = self._validate(value)

    def __delitem__(self, index):
        del self._items[index]

    def insert(self, index, value):
        self._items.insert(index, self._validate(value))

    def extend(self, values):
        proposed = [self._validate(item) for item in values]
        self._items.extend(proposed)

    def __repr__(self):
        return f'CheckedStringList({self._items!r})'

### Step 3 — Let the descriptor manage ownership and replacement

Each instance gets its own controlled list on first read. Replacing the attribute constructs a *new* checked list, so a caller cannot mutate the instance indirectly through the original list. A failed replacement never reaches the final dictionary write.

In [28]:
class CheckedListField:
    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        if self.name not in instance.__dict__:
            instance.__dict__[self.name] = CheckedStringList()
        return instance.__dict__[self.name]

    def __set__(self, instance, value):
        candidate = CheckedStringList(value)
        instance.__dict__[self.name] = candidate

class Document:
    tags = CheckedListField()

a_doc, b_doc = Document(), Document()
assert a_doc.tags is not b_doc.tags
original = ['a', 'b']
a_doc.tags = original
original.append('not forwarded')
assert list(a_doc.tags) == ['a', 'b']
print(a_doc.tags)

CheckedStringList(['a', 'b'])


### Step 4 — Test the mutation surface, especially bulk rollback

These are the critical tests a simple `isinstance(value, list)` field would miss. Notice how the list itself is still mutable; we have changed *which object* owns that mutation and where validation occurs.

In [29]:
a_doc.tags.append('c')
a_doc.tags.insert(0, 'start')
a_doc.tags[1] = 'A'
a_doc.tags[2:4] = ['B', 'C']
assert list(a_doc.tags) == ['start', 'A', 'B', 'C']
before = list(a_doc.tags)
raises(TypeError, lambda: a_doc.tags.extend(['safe', 123]), 'strings')
raises(TypeError, lambda: a_doc.tags.__setitem__(slice(1, 3), ['ok', None]), 'strings')
raises(TypeError, lambda: setattr(a_doc, 'tags', ['valid', 3]), 'strings')
assert list(a_doc.tags) == before
assert list(b_doc.tags) == []
del a_doc.tags[0]
assert list(a_doc.tags) == ['A', 'B', 'C']
print('Bulk operations were all-or-nothing:', a_doc.tags)

Expected TypeError: all elements must be strings
Expected TypeError: all elements must be strings
Expected TypeError: all elements must be strings
Bulk operations were all-or-nothing: CheckedStringList(['A', 'B', 'C'])


**Design review.** The proxy is an API boundary, not a security sandbox: callers can intentionally access `_items`. Python's `MutableSequence` also supplies `+=` via `extend` on this implementation; if adding more mutation methods, inspect them for partial updates. Neither in-place mutation nor direct dictionary edits invoke the outer descriptor's `__set__`. For versioning, auditing, or cross-field invariants, add an explicit callback or owner method for each successful mutation.

## Problem 9 — Why a thread-safe setter does not make `+=` atomic

**The challenge.** A shared counter descriptor guards reads and writes with a per-instance lock. Does `obj.count += 1` then become an atomic increment? Construct a deterministic lost update and provide a safe owner method.

**Success criteria.** Explain the read–modify–write gap, demonstrate the lost update without timing guesses, and validate correct totals with concurrent atomic increments.

We will solve it through small experiments. Try to predict each outcome *before* executing its cell, then compare it with the explanation that follows.

### Step 1 — Create a lock-protected descriptor

The lock belongs to each **instance**, not to the descriptor shared by the entire class. This prevents unrelated counters from unnecessarily contending. Getters and setters use the lock, but each operation releases it before the next attribute operation starts.

In [30]:
import threading

class LockedInteger:
    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        with instance._lock:
            return instance.__dict__[self.name]

    def __set__(self, instance, value):
        if type(value) is not int or value < 0:
            raise ValueError('counter must be a nonnegative exact int')
        with instance._lock:
            instance.__dict__[self.name] = value

class SafeCounter:
    count = LockedInteger()

    def __init__(self):
        self._lock = threading.RLock()
        self.count = 0

    def increment(self, amount=1):
        if type(amount) is not int or amount < 0:
            raise ValueError('increment amount must be nonnegative exact int')
        with self._lock:  # one lock spans the complete read-modify-write operation
            self.count = self.count + amount
            return self.count

### Step 2 — Create a reproducible race using a barrier

Both workers read the value *before either worker is allowed to write*. Locking individual getters and setters cannot prevent this interleaving. The barrier makes the example deterministic rather than relying on the scheduler or the GIL.

In [31]:
counter = SafeCounter()
barrier = threading.Barrier(2)

def unsafe_increment_once():
    old = counter.count
    barrier.wait()
    counter.count = old + 1

threads = [threading.Thread(target=unsafe_increment_once) for _ in range(2)]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()
assert counter.count == 1  # two attempts; one update was lost
print('Two unsafe increments produced:', counter.count)

Two unsafe increments produced: 1


### Step 3 — Guard the full operation and test parallel workloads

`SafeCounter.increment` uses an `RLock` so the owner method may safely call the descriptor’s own locked accessor/setter while holding the lock. This is a practical example of why lock scope must match the invariant being protected.

In [32]:
from concurrent.futures import ThreadPoolExecutor

counter = SafeCounter()
def worker(repetitions):
    for _ in range(repetitions):
        counter.increment()

with ThreadPoolExecutor(max_workers=6) as pool:
    list(pool.map(worker, [250] * 6))
assert counter.count == 1500
raises(ValueError, lambda: counter.increment(True), 'exact int')
assert counter.count == 1500
print('Atomic increments produced:', counter.count)

Expected ValueError: increment amount must be nonnegative exact int
Atomic increments produced: 1500


**Design review.** `obj.count += 1` still comprises a getter, arithmetic, and a setter; it does not call `increment`. A reentrant lock matters here because the owner method invokes descriptor methods that also acquire the same lock. Real concurrency code may also need timeouts, cancellation behavior, and careful lock ordering if multiple objects participate.

## Problem 10 — Migrate an old storage key on first access

**The challenge.** An older serialized model stored `"legacy_title"`, but the new API exposes `title`. Write a descriptor that validates and migrates old data lazily without losing the old value when validation fails.

**Success criteria.** Perform one-time migration on a read; prioritize the modern key; validate before removing the legacy key; make direct writes remove stale legacy data.

We will solve it through small experiments. Try to predict each outcome *before* executing its cell, then compare it with the explanation that follows.

### Step 1 — Decide what “migration” means at the instance level

This example handles *an in-memory instance dictionary*: it does not read a database or promise durable persistence. A modern key takes precedence. A legacy value is validated, copied to the modern key, and only then removed.

In [33]:
class MigratingText:
    def __init__(self, old_key):
        self.old_key = old_key

    def __set_name__(self, owner, name):
        if name == self.old_key:
            raise ValueError('old and new keys must be distinct')
        self.name = name

    def _validate(self, value):
        if not isinstance(value, str) or not value.strip():
            raise ValueError(f'{self.name} must be a nonblank string')
        return value.strip()

    def __get__(self, instance, owner):
        if instance is None:
            return self
        data = instance.__dict__
        if self.name in data:
            return data[self.name]
        if self.old_key not in data:
            raise AttributeError(self.name)
        candidate = self._validate(data[self.old_key])
        data[self.name] = candidate
        del data[self.old_key]
        return candidate

    def __set__(self, instance, value):
        candidate = self._validate(value)
        instance.__dict__[self.name] = candidate
        instance.__dict__.pop(self.old_key, None)

class Article:
    title = MigratingText('legacy_title')

### Step 2 — Test lazy behavior and idempotence

Nothing migrates merely because the class exists. Only reading the managed public attribute triggers the one-time conversion. Repeated reads should not keep modifying the stored dictionary.

In [34]:
article = Article()
article.__dict__['legacy_title'] = '  Original heading  '
assert 'title' not in article.__dict__
assert article.title == 'Original heading'
assert article.__dict__ == {'title': 'Original heading'}
first_snapshot = article.__dict__.copy()
assert article.title == 'Original heading'
assert article.__dict__ == first_snapshot
print('One successful lazy migration:', article.__dict__)

One successful lazy migration: {'title': 'Original heading'}


### Step 3 — Demonstrate why validation must happen before deletion

A corrupt legacy value should not disappear simply because a read failed. Similarly, an invalid modern assignment should neither overwrite the current valid title nor discard the legacy value.

In [35]:
corrupt = Article()
corrupt.__dict__['legacy_title'] = 404
raises(ValueError, lambda: corrupt.title, 'nonblank')
assert corrupt.__dict__ == {'legacy_title': 404}
raises(ValueError, lambda: setattr(corrupt, 'title', '   '), 'nonblank')
assert corrupt.__dict__ == {'legacy_title': 404}
corrupt.title = 'New title'
assert corrupt.__dict__ == {'title': 'New title'}
print('Bad legacy data was preserved until a valid replacement arrived.')

Expected ValueError: title must be a nonblank string
Expected ValueError: title must be a nonblank string
Bad legacy data was preserved until a valid replacement arrived.


**Design review.** A getter with a migration side effect can surprise callers who expect reads to be pure. Document it, and consider an explicit migration method for systems with persistence, locks, audit trails, or complex compatibility requirements. The example demonstrates careful ordering within an in-memory mapping, not a crash-safe database transaction.

## Problem 11 — Delegate storage to a nested settings dictionary

**The challenge.** Some model attributes belong in a dedicated per-instance configuration mapping instead of directly in `__dict__`. Make several descriptors share that mapping without sharing values among owner instances.

**Success criteria.** Provide typed get/set/delete operations, missing-value semantics, independence between models, and predictable handling of invalid assignments.

We will solve it through small experiments. Try to predict each outcome *before* executing its cell, then compare it with the explanation that follows.

### Step 1 — Separate the location of storage from field semantics

The descriptor declares which key it owns and which exact type it accepts. The *owner* creates its `_settings` dictionary. The descriptor object never holds an individual user’s setting value.

In [36]:
class SettingsEntry:
    def __init__(self, expected_type):
        self.expected_type = expected_type

    def __set_name__(self, owner, name):
        self.name = name

    def _mapping(self, instance):
        return instance._settings

    def __get__(self, instance, owner):
        if instance is None:
            return self
        mapping = self._mapping(instance)
        try:
            return mapping[self.name]
        except KeyError:
            raise AttributeError(f'{self.name} is unset') from None

    def __set__(self, instance, value):
        if type(value) is not self.expected_type:
            raise TypeError(f'{self.name} must be exactly {self.expected_type.__name__}')
        self._mapping(instance)[self.name] = value

    def __delete__(self, instance):
        mapping = self._mapping(instance)
        if self.name not in mapping:
            raise AttributeError(f'{self.name} is unset')
        del mapping[self.name]

class AppPreferences:
    dark_mode = SettingsEntry(bool)
    max_items = SettingsEntry(int)

    def __init__(self):
        self._settings = {}

### Step 2 — Test the public contract and inspect the private mapping

The expected type is exact: an integer `1` is *not* a boolean setting, while `True` is not an integer `max_items`. Each owner constructs its own dictionary, so one owner’s settings cannot leak to another through descriptor state.

In [37]:
prefs1, prefs2 = AppPreferences(), AppPreferences()
prefs1.dark_mode = True
prefs1.max_items = 25
prefs2.dark_mode = False
assert prefs1._settings == {'dark_mode': True, 'max_items': 25}
assert prefs2._settings == {'dark_mode': False}
assert not hasattr(prefs2, 'max_items')
raises(TypeError, lambda: setattr(prefs1, 'dark_mode', 1), 'bool')
raises(TypeError, lambda: setattr(prefs1, 'max_items', True), 'int')
assert prefs1.max_items == 25
del prefs1.max_items
assert not hasattr(prefs1, 'max_items')
print('Separate mappings:', prefs1._settings, prefs2._settings)

Expected TypeError: dark_mode must be exactly bool
Expected TypeError: max_items must be exactly int
Separate mappings: {'dark_mode': True} {'dark_mode': False}


### Step 3 — Make the encapsulation limitation explicit

The public descriptor path validates. Direct modification of the backing dictionary bypasses it, just as direct writes to `__dict__` bypass an ordinary validating descriptor. If this matters, do not expose the mutable mapping to untrusted callers.

In [38]:
unsafe = AppPreferences()
unsafe._settings['max_items'] = 'not an integer'
assert unsafe.max_items == 'not an integer'
print('Direct backing-store edits bypassed descriptor validation.')

Direct backing-store edits bypassed descriptor validation.


**Design review.** Dedicated storage can be a useful integration seam for configuration files, adapters, and domain-specific containers. It does not automatically provide persistence, encryption, or security; those require explicit separate designs. A robust model can make the backing storage private by convention and validate data again when importing it from external sources.

## Problem 12 — Descriptors and the method resolution order (MRO)

**The challenge.** Two parent classes define a descriptor with the same name. A child inherits from both, and a grandchild shadows the name with a plain class attribute. Which getter runs, and can a descriptor’s data-descriptor status defeat class-level shadowing?

**Success criteria.** Predict class MRO lookup, contrast instance dictionary precedence, demonstrate `super()` lookup, and explain the maintainability risk of same-name base fields.

We will solve it through small experiments. Try to predict each outcome *before* executing its cell, then compare it with the explanation that follows.

### Step 1 — Build descriptors that reveal exactly which base won

We implement `__set__`, so these are data descriptors. Each stores its value under a unique private key. This makes it possible to tell whether the left-base or right-base descriptor handled an operation.

In [39]:
class OriginField:
    def __init__(self, label):
        self.label = label
        self.key = f'_value_from_{label}'

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return (self.label, instance.__dict__.get(self.key))

    def __set__(self, instance, value):
        instance.__dict__[self.key] = value

class LeftBase:
    token = OriginField('left')

class RightBase:
    token = OriginField('right')

class Combined(LeftBase, RightBase):
    pass

print('MRO:', [cls.__name__ for cls in Combined.__mro__])

MRO: ['Combined', 'LeftBase', 'RightBase', 'object']


### Step 2 — See why the leftmost base wins normal lookup

The first matching class attribute found along the MRO is the left-base descriptor. Data-descriptor precedence applies *after locating* the class attribute; it does not force Python to collect all descriptors with the same name from every base.

In [40]:
combined = Combined()
combined.token = 'hello'
assert combined.token == ('left', 'hello')
assert combined.__dict__ == {'_value_from_left': 'hello'}
assert Combined.token is LeftBase.token
print('Normal access chooses:', combined.token)

Normal access chooses: ('left', 'hello')


### Step 3 — Use `super()` to inspect a different section of the MRO

`super(LeftBase, combined)` starts the search **after** `LeftBase`; it therefore finds `RightBase.token`. This is an explicit method-resolution operation, not evidence that the right-side descriptor participates in ordinary `combined.token` access.

In [41]:
right_view = super(LeftBase, combined).token
assert right_view == ('right', None)
RightBase.__dict__['token'].__set__(combined, 'alternate')
assert super(LeftBase, combined).token == ('right', 'alternate')
assert combined.token == ('left', 'hello')
print('Different MRO sections:', combined.token, super(LeftBase, combined).token)

Different MRO sections: ('left', 'hello') ('right', 'alternate')


### Step 4 — Show that a subclass class attribute shadows an inherited descriptor

A plain attribute declared *on a subclass* is found before inherited descriptors. Its status as a non-descriptor is irrelevant once it wins MRO lookup. An assignment to the instance then creates its own ordinary instance attribute.

In [42]:
class Shadowed(Combined):
    token = 'class default'

shadowed = Shadowed()
assert shadowed.token == 'class default'
shadowed.token = 'instance override'
assert shadowed.token == 'instance override'
assert shadowed.__dict__ == {'token': 'instance override'}
assert isinstance(inspect.getattr_static(Combined, 'token'), OriginField)
assert inspect.getattr_static(Shadowed, 'token') == 'class default'
print('Subclass shadowing:', shadowed.__dict__)

Subclass shadowing: {'token': 'instance override'}


**Design review.** Descriptors are class-level APIs, not universal mechanisms that supersede all inheritance rules. Avoid giving unrelated bases the same managed field name unless their MRO behavior is deliberate. If inheritance must combine configuration, use explicit names, explicit delegation, or a class-creation validation policy rather than relying on accidental MRO order.

---

## Integration exercise A — Make normalization and a transition policy coexist

**Problem.** Reuse the **new** text pipeline and state machine together in a single model. Give it a case-insensitive, nonblank title and a workflow state. Confirm that changing the title cannot change the state and that an illegal state transition cannot change the title.

**Plan.** No new descriptor class is required. Composition at the owner level is the simplest answer: declare the two independent fields and write a short constructor. We keep this exercise focused on per-field independence; if a business rule couples the two fields, use an explicit owner operation and an appropriate transaction boundary.

In [43]:
class EditorialItem:
    title = TextPipeline(normalizers=(nfc, str.strip, str.casefold),
                         validators=(nonblank, at_most(40)))
    state = StateField(TRANSITIONS)

    def __init__(self, title):
        self.title = title
        self.state = 'draft'

item = EditorialItem('  PYTHON Descriptors  ')
assert (item.title, item.state) == ('python descriptors', 'draft')
raises(ValueError, lambda: setattr(item, 'state', 'approved'), 'illegal transition')
raises(ValueError, lambda: setattr(item, 'title', '   '), 'blank')
assert (item.title, item.state) == ('python descriptors', 'draft')
print('Both fields preserved their contracts.')

Expected ValueError: illegal transition: 'draft' -> 'approved'
Expected ValueError: value cannot be blank
Both fields preserved their contracts.


**Why this is a solution, not just a demonstration.** Each field's descriptor owns a single rule and a single committed storage location. Reuse here means composing field definitions, not creating a complicated “everything validator” descriptor. Note that the constructor sets `title` and `state` sequentially; these assertions establish normal behavior and failure atomicity of *individual* later assignments, not an all-or-nothing constructor transaction.

## Integration exercise B — Add a focused regression test to the mutable-sequence solution

**Problem.** Python's sequence interface also allows stepped slice replacement (`seq[::2] = ...`). We should preserve standard list semantics: when the number of replacement items is wrong, Python raises `ValueError` and the original list stays untouched. Verify that the controlled list's existing implementation already has that property.

**Plan.** Validation builds a complete replacement list before calling the built-in list slice assignment. The built-in operation checks its length constraint before committing.

In [44]:
sequence = CheckedStringList(['a', 'b', 'c', 'd'])
snapshot = list(sequence)
raises(ValueError, lambda: sequence.__setitem__(slice(None, None, 2), ['only-one']))
assert list(sequence) == snapshot
sequence[::2] = ['A', 'C']
assert list(sequence) == ['A', 'b', 'C', 'd']
print('Stepped-slice semantics preserved:', sequence)

Expected ValueError: attempt to assign sequence of size 1 to extended slice of size 2
Stepped-slice semantics preserved: CheckedStringList(['A', 'b', 'C', 'd'])


**Extension question.** What should happen if a validator has a side effect and the built-in list later rejects the replacement length? Our list remains unchanged, but those outside side effects are not undone. This is why pure validation functions simplify correct designs.

## Integration exercise C — Make an invariant checklist executable

**Problem.** Create a single short smoke test for representative success and failure behavior across four unrelated descriptor designs. This mimics a useful practice in a real codebase: after a refactor, prove each descriptor still honors its public contract.

**Plan.** Test successful normal access, a specific rejected input, post-failure state, and instance isolation. We avoid relying on the earlier demonstration objects so the test can run repeatedly without shared mutable fixtures.

In [45]:
def smoke_test():
    p = ProfileText()
    p.handle = '  ALICE  '
    raises(ValueError, lambda: setattr(p, 'handle', ' '), 'blank')
    assert p.handle == 'alice'

    money = InvoiceLine()
    money.unit_price = '4.20'
    raises(TypeError, lambda: setattr(money, 'unit_price', 4.2), 'Decimal')
    assert money.unit_price == Decimal('4.20')

    one, two = Document(), Document()
    one.tags.append('only-one')
    assert list(two.tags) == []

    flow = ReviewRequest()
    raises(ValueError, lambda: setattr(flow, 'status', 'approved'), 'illegal')
    assert flow.status == 'draft'
    return 'Four descriptor contracts passed.'

print(smoke_test())

Expected ValueError: value cannot be blank
Expected TypeError: unit_price: use Decimal, int, or decimal string
Expected ValueError: illegal transition: 'draft' -> 'approved'
Four descriptor contracts passed.


---

## Final concept map: how to choose a descriptor design

| Requirement | Mechanism demonstrated | Important limitation |
|---|---|---|
| Reusing a descriptor *object* accidentally | Defensive cloning before `__set_name__` | Cloning must respect descriptor-specific state |
| Clean and consistent text | Transform → validate → commit | Individual field update, not a database transaction |
| Unset distinct from `None` | Raise `AttributeError` for missing | Dynamic lookup may have side effects |
| Several related validators | Base `validate()` + `super()` | Subclasses could bypass hooks if badly designed |
| Slotted owner without `__dict__` | Explicit private slot | Owner must reserve one slot per field |
| Legal workflow changes | Per-instance transition table lookup | Not authorization; concurrent updates need locks |
| Monetary precision | `Decimal`, strict inputs, no silent rounding | Currency, tax, and rounding policy are separate |
| Validate *in-place* list changes | Controlled `MutableSequence` | Direct access to internal list can bypass the API |
| Atomic increments | Lock spans full read-modify-write | Locking getter and setter individually is insufficient |
| Legacy storage compatibility | Validate → copy → remove on first read | Not a persistent crash-safe migration |
| Values in nested storage | Per-instance delegated mapping | Raw mapping access bypasses validation |
| Conflicting inherited fields | MRO and explicit introspection | Subclass class attribute can shadow inherited descriptor |

### Closing reflection

The supplied starting point answers “How do I replace repeated `Int`, `Float`, and `List` setters with a generic validator?” These new worked problems ask what happens **after reuse**: binding names, API boundaries, storage schemes, state transitions, observability, concurrency, and inheritance. A strong descriptor makes its invariants explicit, stores per-object state correctly, and verifies edge cases with small executable experiments.

**Notebook execution contract:** code cells should execute in order, with expected errors caught by the helper. Assertions are intentionally active and should not be run with Python optimization that disables them.